# 🚀 Tank Battle 2D — High-Performance Cloud Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BohatyrovIvanUcu/tank_battle_playground/blob/main/colab_trainer.ipynb)

Цей блокнот налаштований для максимальної продуктивності на **Google Colab (A100 GPU / High-RAM CPU з 12–16 ядрами)**.

### 🏆 Стратегія тренування чемпіона (2 етапи):
1. **Етап 1 (Базовий вишкіл — 100 поколінь проти `rule_based` бота):** танк вчиться основам: не врізатися в стіни, переслідувати ворога, наводити дуло за кутом $\sin(\text{turret}) / \cos(\text{turret})$ та стріляти впритул без промахів.
2. **Етап 2 (Коеволюція / Self-Play — 4000 поколінь проти самого себе):** танки безперервно б'ються один з одним та з «Залою Слави» минулих чемпіонів, виробляючи складні тактики, обманні маневри та снайперське ведення вогню на випередження.

Завдяки багатоядерній паралелізації (`--workers 16`) увесь цикл 4100 поколінь пролітає приблизно за **10–12 хвилин**!

## 1. 🖥️ Перевірка заліза (GPU & CPU)

In [ ]:
# Перевірка відеокарти та кількості ядер процесора
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):|Thread|Core"

import multiprocessing as mp
print(f"\n[+] Доступно ядер CPU для паралельного тренування: {mp.cpu_count()}")

## 2. 🦀 Встановлення Rust, Maturin та залежностей (~30 сек)

In [ ]:
# Встановлюємо офіційний компілятор Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

import os
import sys
os.environ["PATH"] = f"/root/.cargo/bin:{os.environ['PATH']}"
!pip install --quiet maturin numpy matplotlib pillow imageio
!rustc --version

## 3. 📦 Клонування репозиторію та збірка рушія

In [ ]:
# Клонуємо свіжу версію репозиторію
!rm -rf tank_battle_playground
!git clone https://github.com/BohatyrovIvanUcu/tank_battle_playground.git
%cd tank_battle_playground

# Компілюємо нативний CPython рушій у релізному режимі (Rust)
!maturin develop --release

# Додаємо шлях до пакетів у sys.path та PYTHONPATH (запобігає ModuleNotFoundError: tank_env)
import sys
import os
for p in ["python", "/content/tank_battle_playground/python"]:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)
os.environ["PYTHONPATH"] = f"{os.path.abspath('python')}:{os.environ.get('PYTHONPATH', '')}"
print("[✓] Білд завершено, оточення налаштовано успішно!")

## 4. ⚔️ Двоетапне тренування: 100 поколінь проти бота + 4000 поколінь Self-Play

Використовуємо всі доступні ядра (`--workers 16`), завдяки чому кожне покоління рахується за частки секунди.

In [ ]:
import multiprocessing as mp
import os

WORKERS = mp.cpu_count()    # Використовувати всі ядра A100 (12-16 ядер)
POPULATION = 30             # 30 особин у популяції
STAGE1_GENS = 100           # 100 поколінь проти rule_based бота
STAGE2_GENS = 4000          # 4000 поколінь Self-Play (бій проти себе)
CHECKPOINT_INTERVAL = 200   # Зберігати зрізи кожні 200 поколінь

print("=" * 65)
print(f" 🥊 ЕТАП 1: Базовий вишкіл ({STAGE1_GENS} поколінь проти rule_based)")
print("=" * 65)
!python3 python/train_ga.py --adapter combat --opponent rule_based --generations {STAGE1_GENS} --population {POPULATION} --workers {WORKERS} --checkpoint-interval {STAGE1_GENS}

print("\n" + "=" * 65)
print(f" ⚔️ ЕТАП 2: Коеволюція / Self-Play ({STAGE2_GENS} поколінь проти самого себе)")
print("=" * 65)
!python3 python/train_ga.py --adapter combat --opponent self --resume checkpoints/best_model.npz --generations {STAGE2_GENS} --start-gen {STAGE1_GENS} --population {POPULATION} --workers {WORKERS} --checkpoint-interval {CHECKPOINT_INTERVAL}

## 5. 🎬 Візуалізація бою прямо в блокноті (Headless GIF Player)
Тут рендериться 2D-поєдинок вашого фінального агента проти базового бота. Анімація відтворюється безпосередньо у Colab!

In [ ]:
import math
import os
import sys

# Гарантуємо наявність шляху до python/tank_env
for p in ["python", "/content/tank_battle_playground/python"]:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)

from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display
from tank_env import TankEnvironment, load_agent

def render_match_gif(agent_a_path, agent_b_path, seed=42, max_steps=400, output_gif="match.gif"):
    env = TankEnvironment(max_steps=max_steps)
    a1 = load_agent(agent_a_path)
    a2 = load_agent(agent_b_path)
    obs1, obs2 = env.reset(seed=seed)

    width, height = 800, 533
    scale_x, scale_y = width / 1200.0, height / 800.0

    frames = []
    for step in range(max_steps):
        act1 = a1.act(obs1)
        act2 = a2.act(obs2)
        obs1, obs2, r, done, info = env.step(act1, act2)

        # Рендеримо кожен 2-й тік
        if step % 2 == 0:
            img = Image.new("RGB", (width, height), "#141923")
            draw = ImageDraw.Draw(img)

            # Перешкоди
            for o in obs1.get("obstacles", []):
                ox, oy, ow, oh = o[0] * 1200 * scale_x, o[1] * 800 * scale_y, o[2] * 1200 * scale_x, o[3] * 800 * scale_y
                draw.rectangle([ox, oy, ox + ow, oy + oh], fill="#2d3748", outline="#4a5568", width=2)

            # Танк 1 (Синій)
            t1_x, t1_y = obs1["x"] * 1200 * scale_x, obs1["y"] * 800 * scale_y
            r1 = 18.0 * scale_x
            draw.ellipse([t1_x - r1, t1_y - r1, t1_x + r1, t1_y + r1], fill="#3182ce", outline="#63b3ed", width=2)
            turr1 = obs1["turret_angle"]
            draw.line([t1_x, t1_y, t1_x + math.cos(turr1) * 26 * scale_x, t1_y + math.sin(turr1) * 26 * scale_y], fill="#63b3ed", width=3)

            # Танк 2 (Червоний)
            t2_x, t2_y = obs2["x"] * 1200 * scale_x, obs2["y"] * 800 * scale_y
            draw.ellipse([t2_x - r1, t2_y - r1, t2_x + r1, t2_y + r1], fill="#e53e3e", outline="#feb2b2", width=2)
            turr2 = obs2["turret_angle"]
            draw.line([t2_x, t2_y, t2_x + math.cos(turr2) * 26 * scale_x, t2_y + math.sin(turr2) * 26 * scale_y], fill="#feb2b2", width=3)

            # Кулі
            for b in obs1.get("bullets", []):
                bx = (obs1["x"] + b[0]) * 1200 * scale_x
                by = (obs1["y"] + b[1]) * 800 * scale_y
                draw.ellipse([bx - 3, by - 3, bx + 3, by + 3], fill="#ecc94b")

            # HP смужки
            draw.rectangle([20, 20, 20 + obs1['hp'] * 2, 32], fill="#3182ce")
            draw.rectangle([width - 220, 20, width - 220 + obs2['hp'] * 2, 32], fill="#e53e3e")

            frames.append(img)

        if done:
            break

    frames[0].save(output_gif, save_all=True, append_images=frames[1:], duration=32, loop=0)
    print(f"[+] Поєдинок збережено в {output_gif}! Кроків: {step}, HP1: {obs1['hp']}, HP2: {obs2['hp']}")
    return output_gif

# Рендеримо бій нашого фінального агента проти Rule-Based бота
gif_path = render_match_gif("checkpoints/agent.py", "rule_based", seed=99)
display(IPyImage(gif_path))

## 6. 📥 Завантаження натренованого чемпіона на ваш комп'ютер
Ця клітинка збереже найкращу модель у `.zip` і автоматично завантажить її через браузер.

In [ ]:
!zip -r tank_champion_4100.zip checkpoints/best_model.npz checkpoints/agent.py

from google.colab import files
files.download("tank_champion_4100.zip")
print("[✓] Архів tank_champion_4100.zip готовий! Просто розархівуйте його в папку checkpoints/ на вашому ПК.")